In [14]:
import polars as pl
import numpy as np
import pandas as pd
import optuna
import pickle
import os
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [15]:
# ------------------------------
# 1. Загрузка и предобработка
# ------------------------------
train = pl.read_parquet('data/train_main_features.parquet')
test = pl.read_parquet('data/test_main_features.parquet')
target = pl.read_parquet('data/train_target.parquet')

In [16]:
cat_features = [col for col in train.columns if col.startswith("cat_feature")]

def preprocess(df):
    df = df.clone()
    num_cols = [col for col in df.columns if col.startswith("num_feature")]
    for col in num_cols:
        median_val = df[col].median()
        df = df.with_columns(pl.col(col).fill_null(median_val))
    for col in cat_features:
        df = df.with_columns(pl.col(col).fill_null(-1).cast(pl.Int32))
    return df

train = preprocess(train)
test = preprocess(test)

target_cols = [col for col in target.columns if col.startswith("target")]
y_all = target[target_cols].to_pandas()
X = train.drop("customer_id").to_pandas()

In [17]:
# ------------------------------
# 2. Подвыборка для оптимизации
# ------------------------------
X_sample, _, y_sample, _ = train_test_split(
    X, y_all, test_size=0.8, random_state=42,
    stratify=y_all['target_1_1']
)
print(f"Размер выборки для оптимизации: {X_sample.shape[0]}")

X_opt_train, X_opt_val, y_opt_train, y_opt_val = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42,
    stratify=y_sample['target_1_1']
)

Размер выборки для оптимизации: 150000


In [18]:
# ------------------------------
# 3. Функция оптимизации
# ------------------------------
def objective(trial, X_tr, y_tr, X_val, y_val, cat_features, target_name):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 400, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'depth': trial.suggest_int('depth', 3, 6),
        'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 7),
        'random_strength': trial.suggest_int('random_strength', 1, 5),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.5, 1.2),
        'border_count': trial.suggest_int('border_count', 32, 128, step=32),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 30),
        'early_stopping_rounds': 30,
        'eval_metric': 'AUC',
        'loss_function': 'Logloss',
        'verbose': False,
        'random_seed': 42,
        'task_type': 'CPU',
        'thread_count': 4
    }
    
    y_tr_target = y_tr[target_name]
    pos = y_tr_target.sum()
    neg = len(y_tr_target) - pos
    if pos > 0 and neg > 0:
        params['class_weights'] = {0: 1, 1: neg / pos}
    
    train_pool = Pool(X_tr, label=y_tr_target, cat_features=cat_features)
    val_pool = Pool(X_val, label=y_val[target_name], cat_features=cat_features)
    
    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=val_pool, verbose=False)
    
    pred_val = model.predict_proba(val_pool)[:, 1]
    auc = roc_auc_score(y_val[target_name], pred_val)
    return auc

In [ ]:
# ------------------------------
# 4. Разбиение на 4 группы
# ------------------------------
num_groups = 4
group_size = len(target_cols) // num_groups
groups = [target_cols[i*group_size:(i+1)*group_size] for i in range(num_groups)]
# Добавляем остаток в последнюю группу
if len(groups[-1]) < len(target_cols) - (num_groups-1)*group_size:
    groups[-1].extend(target_cols[num_groups*group_size:])
print("Группы целевых переменных:")
for i, g in enumerate(groups):
    print(f"  Группа {i+1}: {len(g)} классов")

Группы целевых переменных:
  Группа 1: 10 классов
  Группа 2: 10 классов
  Группа 3: 10 классов
  Группа 4: 11 классов


In [24]:
best_params = {}
n_trials_per_class = 6   # сокращённое количество попыток

In [25]:
#group 1
for target_name in groups[0]:
    print(f"\nОптимизация для {target_name}...")
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(
        lambda trial: objective(trial, X_opt_train, y_opt_train, X_opt_val, y_opt_val,
                                cat_features, target_name),
        n_trials=n_trials_per_class,
        show_progress_bar=True
    )
    best_params[target_name] = study.best_params
    print(f"Лучший AUC: {study.best_value:.4f}, параметры: {study.best_params}")

[I 2026-03-21 22:09:04,065] A new study created in memory with name: no-name-5f4f9f6d-131f-477d-9d23-e4959b4a9d8a



Оптимизация для target_1_1...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 22:10:04,431] Trial 0 finished with value: 0.8893075053202882 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.8893075053202882.
[I 2026-03-21 22:10:54,465] Trial 1 finished with value: 0.8786670907350982 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.8893075053202882.
[I 2026-03-21 22:11:34,029] Trial 2 finished with value: 0.8529882789558416 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.88930

[I 2026-03-21 22:12:40,609] A new study created in memory with name: no-name-ee62a81f-40ab-4301-817f-5ae2399e67d5


...

[I 2026-03-21 22:32:57,001] Trial 0 finished with value: 0.6748720626250528 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.6748720626250528.
[I 2026-03-21 22:33:11,590] Trial 1 finished with value: 0.638456353784814 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.6748720626250528.
[I 2026-03-21 22:33:48,508] Trial 2 finished with value: 0.6480612960231339 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.674872

[I 2026-03-21 22:34:19,574] A new study created in memory with name: no-name-d0aec076-63d9-4640-a1bb-81c71f94ca00


[I 2026-03-21 22:34:19,572] Trial 5 finished with value: 0.6506980327437893 and parameters: {'iterations': 100, 'learning_rate': 0.04437555550097432, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 2, 'bagging_temperature': 0.9637655990477874, 'border_count': 64, 'min_data_in_leaf': 16}. Best is trial 0 with value: 0.6748720626250528.
Лучший AUC: 0.6749, параметры: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}

Оптимизация для target_2_5...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 22:34:45,571] Trial 0 finished with value: 0.6573673988222034 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.6573673988222034.
[I 2026-03-21 22:34:55,377] Trial 1 finished with value: 0.6433157526185679 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.6573673988222034.
[I 2026-03-21 22:35:07,855] Trial 2 finished with value: 0.6303342902057663 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.65736

In [26]:
#group 2
for target_name in groups[1]:
    print(f"\nОптимизация для {target_name}...")
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(
        lambda trial: objective(trial, X_opt_train, y_opt_train, X_opt_val, y_opt_val,
                                cat_features, target_name),
        n_trials=n_trials_per_class,
        show_progress_bar=True
    )
    best_params[target_name] = study.best_params
    print(f"Лучший AUC: {study.best_value:.4f}, параметры: {study.best_params}")

[I 2026-03-21 22:36:01,905] A new study created in memory with name: no-name-f5eafa0f-3b35-4e02-8e95-24ed96377f82



Оптимизация для target_2_6...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 22:36:34,124] Trial 0 finished with value: 0.6925373789407423 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.6925373789407423.
[I 2026-03-21 22:36:56,400] Trial 1 finished with value: 0.6350845176976074 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.6925373789407423.
[I 2026-03-21 22:37:29,525] Trial 2 finished with value: 0.6321697399499882 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.69253

[I 2026-03-21 22:38:26,969] A new study created in memory with name: no-name-66f25b4f-a6f8-4a7e-b830-a725c795f3a7


...

[I 2026-03-21 22:52:43,964] Trial 0 finished with value: 0.8519561913872464 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.8519561913872464.
[I 2026-03-21 22:53:30,891] Trial 1 finished with value: 0.8521661201272391 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 1 with value: 0.8521661201272391.
[I 2026-03-21 22:54:00,336] Trial 2 finished with value: 0.8442844041722597 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 1 with value: 0.85216

[I 2026-03-21 22:54:45,473] A new study created in memory with name: no-name-226c6a77-2cd1-4e90-8f1a-0212c31b1bc1


[I 2026-03-21 22:54:45,471] Trial 5 finished with value: 0.8537470091860413 and parameters: {'iterations': 100, 'learning_rate': 0.04437555550097432, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 2, 'bagging_temperature': 0.9637655990477874, 'border_count': 64, 'min_data_in_leaf': 16}. Best is trial 4 with value: 0.8569135167002759.
Лучший AUC: 0.8569, параметры: {'iterations': 100, 'learning_rate': 0.09210273435223537, 'depth': 6, 'l2_leaf_reg': 6, 'random_strength': 2, 'bagging_temperature': 0.5683704798044688, 'border_count': 96, 'min_data_in_leaf': 14}

Оптимизация для target_5_1...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 22:55:15,279] Trial 0 finished with value: 0.7296898141914636 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.7296898141914636.
[I 2026-03-21 22:55:40,878] Trial 1 finished with value: 0.7198717056057959 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.7296898141914636.
[I 2026-03-21 22:56:16,843] Trial 2 finished with value: 0.7250318050024809 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.72968

In [27]:
#group 3
for target_name in groups[2]:
    print(f"\nОптимизация для {target_name}...")
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(
        lambda trial: objective(trial, X_opt_train, y_opt_train, X_opt_val, y_opt_val,
                                cat_features, target_name),
        n_trials=n_trials_per_class,
        show_progress_bar=True
    )
    best_params[target_name] = study.best_params
    print(f"Лучший AUC: {study.best_value:.4f}, параметры: {study.best_params}")

[I 2026-03-21 22:57:15,326] A new study created in memory with name: no-name-1e5a9b72-302d-4c89-a515-cc0b41dc6e3a



Оптимизация для target_5_2...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 22:57:43,896] Trial 0 finished with value: 0.6827209899195001 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.6827209899195001.
[I 2026-03-21 22:57:56,647] Trial 1 finished with value: 0.6687506116319407 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.6827209899195001.
[I 2026-03-21 22:58:04,831] Trial 2 finished with value: 0.66111039740068 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.6827209

[I 2026-03-21 22:58:26,941] A new study created in memory with name: no-name-c93508b4-a38b-4515-95c8-efb62b480c6f


...

[I 2026-03-21 23:17:28,207] Trial 0 finished with value: 0.7151739476025145 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.7151739476025145.
[I 2026-03-21 23:17:35,970] Trial 1 finished with value: 0.6647827910149772 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.7151739476025145.
[I 2026-03-21 23:17:47,468] Trial 2 finished with value: 0.6674665586900874 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.71517

[I 2026-03-21 23:18:51,887] A new study created in memory with name: no-name-6d029f9f-3a84-44e2-9ca8-b02899f5e6aa


[I 2026-03-21 23:18:51,885] Trial 5 finished with value: 0.6907577163933101 and parameters: {'iterations': 100, 'learning_rate': 0.04437555550097432, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 2, 'bagging_temperature': 0.9637655990477874, 'border_count': 64, 'min_data_in_leaf': 16}. Best is trial 3 with value: 0.7426294691704824.
Лучший AUC: 0.7426, параметры: {'iterations': 250, 'learning_rate': 0.07076922519051033, 'depth': 3, 'l2_leaf_reg': 4, 'random_strength': 3, 'bagging_temperature': 0.5325152889039984, 'border_count': 96, 'min_data_in_leaf': 6}

Оптимизация для target_8_1...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 23:19:53,229] Trial 0 finished with value: 0.9637273766257822 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.9637273766257822.
[I 2026-03-21 23:20:44,968] Trial 1 finished with value: 0.9611029073003421 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.9637273766257822.
[I 2026-03-21 23:21:30,302] Trial 2 finished with value: 0.9559356851789061 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.96372

In [28]:
#group 4
for target_name in groups[3]:
    print(f"\nОптимизация для {target_name}...")
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(
        lambda trial: objective(trial, X_opt_train, y_opt_train, X_opt_val, y_opt_val,
                                cat_features, target_name),
        n_trials=n_trials_per_class,
        show_progress_bar=True
    )
    best_params[target_name] = study.best_params
    print(f"Лучший AUC: {study.best_value:.4f}, параметры: {study.best_params}")

[I 2026-03-21 23:27:38,547] A new study created in memory with name: no-name-5780cc65-2b54-411d-b97b-41e185d16063



Оптимизация для target_8_2...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 23:28:40,168] Trial 0 finished with value: 0.8168028471094473 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.8168028471094473.
[I 2026-03-21 23:28:59,735] Trial 1 finished with value: 0.7409589722874033 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.8168028471094473.
[I 2026-03-21 23:29:40,920] Trial 2 finished with value: 0.7635410042234279 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.81680

[I 2026-03-21 23:30:49,180] A new study created in memory with name: no-name-3b07613b-88ff-4fd3-ac50-2bd4d8269929


...

[I 2026-03-21 23:54:26,140] Trial 0 finished with value: 0.9221648084515299 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.9221648084515299.
[I 2026-03-21 23:55:15,260] Trial 1 finished with value: 0.9232495338922747 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 1 with value: 0.9232495338922747.
[I 2026-03-21 23:55:53,274] Trial 2 finished with value: 0.908874609538125 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 1 with value: 0.923249

[I 2026-03-21 23:56:57,677] A new study created in memory with name: no-name-a0c093e0-7c73-4afa-a9aa-8348dc67bfb9


[I 2026-03-21 23:56:57,675] Trial 5 finished with value: 0.9112279475874829 and parameters: {'iterations': 100, 'learning_rate': 0.04437555550097432, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 2, 'bagging_temperature': 0.9637655990477874, 'border_count': 64, 'min_data_in_leaf': 16}. Best is trial 1 with value: 0.9232495338922747.
Лучший AUC: 0.9232, параметры: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}

Оптимизация для target_10_1...


  0%|          | 0/6 [00:00<?, ?it/s]

[I 2026-03-21 23:57:56,229] Trial 0 finished with value: 0.7386493822511297 and parameters: {'iterations': 200, 'learning_rate': 0.09237421878009823, 'depth': 5, 'l2_leaf_reg': 5, 'random_strength': 1, 'bagging_temperature': 0.6091961642353418, 'border_count': 32, 'min_data_in_leaf': 26}. Best is trial 0 with value: 0.7386493822511297.
[I 2026-03-21 23:58:46,353] Trial 1 finished with value: 0.7327357845932962 and parameters: {'iterations': 300, 'learning_rate': 0.06251028636335225, 'depth': 3, 'l2_leaf_reg': 7, 'random_strength': 5, 'bagging_temperature': 0.6486373774747933, 'border_count': 32, 'min_data_in_leaf': 6}. Best is trial 0 with value: 0.7386493822511297.
[I 2026-03-21 23:59:29,345] Trial 2 finished with value: 0.7277264794471681 and parameters: {'iterations': 200, 'learning_rate': 0.04653920936389926, 'depth': 4, 'l2_leaf_reg': 3, 'random_strength': 4, 'bagging_temperature': 0.5976457024564292, 'border_count': 64, 'min_data_in_leaf': 11}. Best is trial 0 with value: 0.73864

In [31]:
best_params

{'target_1_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_2': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_1_3': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_4': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_5': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l

{'target_1_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_2': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_1_3': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_4': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_1_5': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_2_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_2_2': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_2_3': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_2_4': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_2_5': {'iterations': 100,
  'learning_rate': 0.09210273435223537,
  'depth': 6,
  'l2_leaf_reg': 6,
  'random_strength': 2,
  'bagging_temperature': 0.5683704798044688,
  'border_count': 96,
  'min_data_in_leaf': 14},
 'target_2_6': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_2_7': {'iterations': 100,
  'learning_rate': 0.09210273435223537,
  'depth': 6,
  'l2_leaf_reg': 6,
  'random_strength': 2,
  'bagging_temperature': 0.5683704798044688,
  'border_count': 96,
  'min_data_in_leaf': 14},
 'target_2_8': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_3_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_3_2': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_3_3': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_3_4': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_3_5': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_4_1': {'iterations': 100,
  'learning_rate': 0.09210273435223537,
  'depth': 6,
  'l2_leaf_reg': 6,
  'random_strength': 2,
  'bagging_temperature': 0.5683704798044688,
  'border_count': 96,
  'min_data_in_leaf': 14},
 'target_5_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_5_2': {'iterations': 100,
  'learning_rate': 0.04437555550097432,
  'depth': 3,
  'l2_leaf_reg': 7,
  'random_strength': 2,
  'bagging_temperature': 0.9637655990477874,
  'border_count': 64,
  'min_data_in_leaf': 16},
 'target_6_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_6_2': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_6_3': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_6_4': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_6_5': {'iterations': 100,
  'learning_rate': 0.09210273435223537,
  'depth': 6,
  'l2_leaf_reg': 6,
  'random_strength': 2,
  'bagging_temperature': 0.5683704798044688,
  'border_count': 96,
  'min_data_in_leaf': 14},
 'target_7_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_7_2': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_7_3': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_8_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_8_2': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_8_3': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_9_1': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_9_2': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_9_3': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_9_4': {'iterations': 250,
  'learning_rate': 0.07076922519051033,
  'depth': 3,
  'l2_leaf_reg': 4,
  'random_strength': 3,
  'bagging_temperature': 0.5325152889039984,
  'border_count': 96,
  'min_data_in_leaf': 6},
 'target_9_5': {'iterations': 300,
  'learning_rate': 0.06251028636335225,
  'depth': 3,
  'l2_leaf_reg': 7,
  'random_strength': 5,
  'bagging_temperature': 0.6486373774747933,
  'border_count': 32,
  'min_data_in_leaf': 6},
 'target_9_6': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_9_7': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26},
 'target_9_8': {'iterations': 300,
  'learning_rate': 0.06251028636335225,
  'depth': 3,
  'l2_leaf_reg': 7,
  'random_strength': 5,
  'bagging_temperature': 0.6486373774747933,
  'border_count': 32,
  'min_data_in_leaf': 6},
 'target_10_1': {'iterations': 200,
  'learning_rate': 0.09237421878009823,
  'depth': 5,
  'l2_leaf_reg': 5,
  'random_strength': 1,
  'bagging_temperature': 0.6091961642353418,
  'border_count': 32,
  'min_data_in_leaf': 26}}

In [ ]:
# ------------------------------
# 4. Обучение финальных моделей на всех данных
# ------------------------------
# Разделим все данные на train/val (80/20) для финального обучения
X_all_train, X_all_val, y_all_train, y_all_val = train_test_split(
    X, y_all, test_size=0.2, random_state=42,
    stratify=y_all['target_1_1']
)

models = {}


Обучение финальных моделей на всех данных...


In [33]:
# group 1
for target_name in groups[0]:
    print(f"Обучаем {target_name}...")
    params = best_params[target_name].copy()
    # Увеличиваем iterations для финальной модели
    if 'iterations' in params:
        params['iterations'] = int(params['iterations'] * 1.2)
    params.pop('early_stopping_rounds', None)
    
    y_tr_target = y_all_train[target_name]
    pos = y_tr_target.sum()
    neg = len(y_tr_target) - pos
    if pos > 0 and neg > 0:
        params['class_weights'] = {0: 1, 1: neg / pos}
    
    train_pool = Pool(X_all_train, label=y_tr_target, cat_features=cat_features)
    model = CatBoostClassifier(**params, verbose=False, random_seed=42)
    model.fit(train_pool)
    models[target_name] = model

Обучаем target_1_1...
Обучаем target_1_2...
Обучаем target_1_3...
Обучаем target_1_4...
Обучаем target_1_5...
Обучаем target_2_1...
Обучаем target_2_2...
Обучаем target_2_3...
Обучаем target_2_4...
Обучаем target_2_5...


In [34]:
# group 2
for target_name in groups[1]:
    print(f"Обучаем {target_name}...")
    params = best_params[target_name].copy()
    # Увеличиваем iterations для финальной модели
    if 'iterations' in params:
        params['iterations'] = int(params['iterations'] * 1.2)
    params.pop('early_stopping_rounds', None)
    
    y_tr_target = y_all_train[target_name]
    pos = y_tr_target.sum()
    neg = len(y_tr_target) - pos
    if pos > 0 and neg > 0:
        params['class_weights'] = {0: 1, 1: neg / pos}
    
    train_pool = Pool(X_all_train, label=y_tr_target, cat_features=cat_features)
    model = CatBoostClassifier(**params, verbose=False, random_seed=42)
    model.fit(train_pool)
    models[target_name] = model

Обучаем target_2_6...
Обучаем target_2_7...
Обучаем target_2_8...
Обучаем target_3_1...
Обучаем target_3_2...
Обучаем target_3_3...
Обучаем target_3_4...
Обучаем target_3_5...
Обучаем target_4_1...
Обучаем target_5_1...


In [35]:
# group 3
for target_name in groups[2]:
    print(f"Обучаем {target_name}...")
    params = best_params[target_name].copy()
    # Увеличиваем iterations для финальной модели
    if 'iterations' in params:
        params['iterations'] = int(params['iterations'] * 1.2)
    params.pop('early_stopping_rounds', None)
    
    y_tr_target = y_all_train[target_name]
    pos = y_tr_target.sum()
    neg = len(y_tr_target) - pos
    if pos > 0 and neg > 0:
        params['class_weights'] = {0: 1, 1: neg / pos}
    
    train_pool = Pool(X_all_train, label=y_tr_target, cat_features=cat_features)
    model = CatBoostClassifier(**params, verbose=False, random_seed=42)
    model.fit(train_pool)
    models[target_name] = model

Обучаем target_5_2...
Обучаем target_6_1...
Обучаем target_6_2...
Обучаем target_6_3...
Обучаем target_6_4...
Обучаем target_6_5...
Обучаем target_7_1...
Обучаем target_7_2...
Обучаем target_7_3...
Обучаем target_8_1...


In [36]:
# group 4
for target_name in groups[3]:
    print(f"Обучаем {target_name}...")
    params = best_params[target_name].copy()
    # Увеличиваем iterations для финальной модели
    if 'iterations' in params:
        params['iterations'] = int(params['iterations'] * 1.2)
    params.pop('early_stopping_rounds', None)
    
    y_tr_target = y_all_train[target_name]
    pos = y_tr_target.sum()
    neg = len(y_tr_target) - pos
    if pos > 0 and neg > 0:
        params['class_weights'] = {0: 1, 1: neg / pos}
    
    train_pool = Pool(X_all_train, label=y_tr_target, cat_features=cat_features)
    model = CatBoostClassifier(**params, verbose=False, random_seed=42)
    model.fit(train_pool)
    models[target_name] = model

Обучаем target_8_2...
Обучаем target_8_3...
Обучаем target_9_1...
Обучаем target_9_2...
Обучаем target_9_3...
Обучаем target_9_4...
Обучаем target_9_5...
Обучаем target_9_6...
Обучаем target_9_7...
Обучаем target_9_8...
Обучаем target_10_1...


In [37]:
# ------------------------------
# 5. Предсказание на тесте
# ------------------------------
print("\nФормирование предсказаний...")
X_test = test.drop("customer_id").to_pandas()
test_pool = Pool(X_test, cat_features=cat_features)

predictions = []
for target_name in target_cols:
    model = models[target_name]
    proba = model.predict_proba(test_pool)[:, 1]
    predictions.append(proba)

predictions = np.column_stack(predictions)
predict_cols = [f"predict_{col.replace('target_', '')}" for col in target_cols]
pred_df = pl.DataFrame(predictions, schema=predict_cols)


Формирование предсказаний...


In [ ]:
# ------------------------------
# 6. Сохранение сабмита
# ------------------------------
submit = pl.DataFrame({'customer_id': test['customer_id']}).hstack(pred_df)
submit.write_parquet("data/submit2.parquet")

: 